In [ ]:
from CRNs import *
import numpy as np
import matplotlib.pyplot as plt
import sympy
import networkx as nx
import random

# --- Example usage with new class structure ---

# reaction_strings = ['A+B->AB', 'A+C->AC', 'B+C->BC', 'AB->AC', 'AB->BC', 'AC->BC', 'A->C']
# reaction_strings = ['A+B->AB', 'A+C->AC', 'B+C->BC', 'B->C', 'A->B', 'A->C']

# r_n = ReactionNetwork.from_reaction_strings(
#     reaction_strings=reaction_strings,
#     L=None,
#     seed=42,
#     species_names=['A', 'B', 'C', 'AB', 'AC', 'BC']
# )

# display(r_n.L)



# n_species = 8
# n_complexes = 9
# n_reactions = 8
# force_reverse = True
# L = np.array([[2, 1, 1, 1, 0, 0, 0, 0],
#               [0, 0, 0, 0, 2, 1, 1, 1]])
# n_cons = len(L)
# n_lcs = n_complexes - n_species + n_cons
# print(n_lcs)
# complexes_per_class = [2, 3, 4]
# reactions_per_class = [1, 3, 6]

n_species = 6
n_complexes = 7
n_reactions = 4
force_reverse = True
L = np.array([[2, 1, 1, 0, 0, 0],
              [0, 0, 0, 1, 1, 1]])
n_cons = len(L)
n_lcs = n_complexes - n_species + n_cons
print(n_lcs)
complexes_per_class = [2, 4]
reactions_per_class = [1, 4]

#seed = 30
seed = np.random.randint(0, 1000000)
np.random.seed(seed)
random.seed(seed)


subset_group_ind = None # this is the "upstream" conservation group
print(L)

# Build the reaction network
r_n = ReactionNetwork(
    n_species, 
    n_complexes, n_reactions, n_lcs, 
    L, seed, force_reverse=force_reverse, subset_group_ind=subset_group_ind, 
    #complexes_per_class=complexes_per_class, reactions_per_class=reactions_per_class
)



# 1. Create the simulator
sim = ReactionNetworkSimulator(r_n)

print("Complexes per class:", r_n.complexes_per_class)
print("Reactions per class:", r_n.reactions_per_class)
print("Extra conservations", r_n.n_species - np.linalg.matrix_rank(r_n.Y @ r_n.A) - r_n.n_cons)

# for i, node_assignment in enumerate(r_n.assignments):
#     print(f"Linkage group {i+1}: {node_assignment}")

# symbolic ODEs
symbolic_rhs, species, rates = sim.get_symbolic_rhs()
reduced_rhs, remaining_syms, const_syms, rate_syms = sim.get_symbolic_reduced_rhs()

M_0t1, M_1t0, M_0b1 = count_conservation_group_changes(r_n)
print(M_0t1, M_1t0, M_0b1)

cycles = compute_cycles(r_n)
print(cycles[1])

# # Visualize linkage classes
# sz = 3
# fig, axes = plt.subplots(1, len(r_n.linkage_groups), figsize=(sz*len(r_n.linkage_groups), sz))

# if len(r_n.linkage_groups) == 1:
#     pos = nx.spring_layout(r_n.linkage_groups[0])
#     nx.draw(r_n.linkage_groups[0], pos=pos, with_labels=True, ax=axes, node_size=1000, connectionstyle='arc3,rad=0.2', arrows=True)
#     axes.set_title("Linkage class 1")
# else:
#     for i, (G, ax) in enumerate(zip(r_n.linkage_groups, axes)):
#         pos = nx.spring_layout(G)
#         nx.draw(G, pos=pos, with_labels=True, ax=ax, node_size=1000, connectionstyle='arc3,rad=0.2', arrows=True)
#         ax.set_title(f"Linkage class {i+1}")
# plt.tight_layout()
# plt.show()

# 2. Reduce the ODE system using conservation laws
sim.solve_conservation_laws()
print("Remaining species after reduction:", sim.remaining_species)
print("Conservation constants (symbols):", sim.const_syms)
print("Eliminated species solutions:", sim.elimination_solutions)




3
[[2 1 1 0 0 0]
 [0 0 0 1 1 1]]
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Setting n_reactions to maximum value: 5
Complexes per class: [2 3 2]
Reactions per class: [1 3 1]
Extra conservations 0
0 0 0
[['r2: E -> D', 'r5: F -> E', 'r6: D -> F']]
Remaining species after reduction: ['B', 'C', 'E', 'F']
Conservation constants (symbols): [const0, const1]
Eliminated species solutions: {'A': -B/2 - C/2 + const0/2, 'D': -E - F + const1}


In [339]:
for _ in range(20):
    r_n = ReactionNetwork(
    n_species, 
    n_complexes, n_reactions, n_lcs, 
    L, seed, force_reverse=force_reverse, subset_group_ind=subset_group_ind, 
    #complexes_per_class=complexes_per_class, reactions_per_class=reactions_per_class
)
    cycles = compute_cycles(r_n)
    print(len(cycles[1]))
    # if len(cycles[1]) == 0:
    #     print(cycles)
    #     break







0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [44]:
extra_complexes = r_n.rng.multinomial(3, [1/n_lcs]*n_lcs) if 3 > 0 else np.zeros(n_lcs, dtype=int)
print(extra_complexes)

[3 0]


In [45]:
[1/n_lcs]*n_lcs

[0.5, 0.5]

In [33]:
E_range = 1
B_range = 1
F_range = 0
C0 = 1
beta = 1
reac_rates = generate_thermodynamic_rates(r_n, C0, beta, E_range, B_range, F_range) 
r_n.update_rates(reac_rates)

t_span = (0, 10000)
num_points = 2000  
r_tol = 1e-6    
a_tol = 1e-6


M_0t1, M_1t0, M_0b1 = count_conservation_group_changes(r_n)


sim = ReactionNetworkSimulator(r_n)
sim.solve_conservation_laws()
reduced_rhs, remaining_syms, const_syms, rate_syms = sim.get_symbolic_reduced_rhs()
(dR_dC, dR_dC_func, 
dR_dl, dR_dl_func, 
dR_dk, dR_dk_func,
dR_dC_dk, dR_dC_dk_func, dR_dl_dk, dR_dl_dk_func,
remaining_syms) = sim.get_derivatives()

# E_range = 1
# B_range = 1
# F_range = 5
# rates = generate_thermodynamic_rates(r_n, C0, beta, E_range, B_range, F_range) 
# r_n.update_rates(rates)


rates = np.array([r_n.reactions[r_idx][2] for r_idx in range(len(r_n.reactions))])
flexible_reduced_ode_rhs = sim.make_reduced_rhs_with_conservation_flexible()


l0 = np.exp(np.random.uniform(np.log(0.0001), np.log(1000.0), size=L.shape[0]))  # Sample uniformly in log space between 0.1 and 1000

C_full = generate_positive_initial_concentrations_nnls(L, l0)
_, C_reduced_init = sim.get_const_and_reduced_init(C_full)
sim.make_reduced_rhs_with_conservation(l0)

sol_reduced, C_reduced_final = sim.integrate(
    lambda C: flexible_reduced_ode_rhs(C, l0), C_reduced_init, t_span=t_span,
    num_points=num_points, method='LSODA', rtol=r_tol, atol=a_tol
)

C_full = sim.recover_eliminated_species(l0, C_reduced_final)
dC_dl = sim.dC_dl_func(C_reduced_final, l0, rates, dR_dC_func, dR_dl_func)
dC_dl_full = sim.compute_dC_dk_full(dC_dl)
signs = np.sign(np.round(dC_dl_full, decimals=12)).tolist()


In [34]:
cycles = compute_cycles(r_n)
print(len(cycles[1]))
print(cycles)
reac_vel = rates * r_n.Psi(C_full)
print(reac_vel)
diff = reac_vel[0:-2:2] - reac_vel[1:-1:2]
print(diff)



2
(Matrix([
[ 1,  0],
[-1,  0],
[ 1,  0],
[ 0,  1],
[ 0,  0],
[ 0, -1],
[ 0,  1],
[ 0,  0]]), [['r0: G -> H', 'r3: F -> G', 'r4: H -> F'], ['r6: A -> B+B', 'r11: B+D -> A', 'r12: B+B -> B+D']])
[1.55873330e-02 1.55873330e-02 6.41408326e-03 6.41408326e-03
 3.50421952e-03 3.50421952e-03 9.60084433e-03 9.60084433e-03
 2.46580220e-02 2.46580220e-02 1.23782532e-02 1.23782532e-02
 1.38986160e-02 1.38986160e-02 4.13365111e-05 4.13365111e-05]
[-1.73472348e-18 -8.67361738e-19  0.00000000e+00 -1.38777878e-17
  3.46944695e-18 -5.20417043e-17 -3.98986399e-17]
